<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 1

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df1 = pd.read_csv('Dataset 1.csv')

# Removes rows that don't have HIGH confidence in their sales estimates
df1 = df1[(df1['saleEstimate_confidenceLevel'] == 'HIGH')]

# Drops all unnecessary columns
df1 = df1.drop(columns=['fullAddress', 'postcode', 'rentEstimate_lowerPrice', 'rentEstimate_currentPrice', 'rentEstimate_upperPrice', 
                        'saleEstimate_lowerPrice', 'saleEstimate_upperPrice', 'saleEstimate_ingestedAt', 'saleEstimate_valueChange.numericChange', 
                        'saleEstimate_valueChange.percentageChange', 'saleEstimate_valueChange.saleDate', 'history_date', 'history_price', 'history_percentageChange', 
                        'history_numericChange', 'saleEstimate_confidenceLevel'])

# Drops all with null values
df1.dropna(inplace=True)

# Remove the top 1% highest prices to reduce high-end outliers before log-transform
upper_quantile = df1['saleEstimate_currentPrice'].quantile(0.99)
df1 = df1[df1['saleEstimate_currentPrice'] <= upper_quantile]

# Apply log transform to the target
df1['saleEstimate_currentPrice'] = np.log(df1['saleEstimate_currentPrice'])

# Create a copy of df1 for Linear Regression before feature engineering
df1lr = df1.copy()

# Apply location-based feature engineering to df1lr only (for Linear Regression)
df1lr['lat_lon_interaction'] = df1lr['latitude'] * df1lr['longitude']
center_lat = df1lr['latitude'].mean()
center_lon = df1lr['longitude'].mean()
df1lr['dist_to_center'] = np.sqrt((df1lr['latitude'] - center_lat)**2 + (df1lr['longitude'] - center_lon)**2)
df1lr = df1lr.drop(columns=['latitude', 'longitude'])  # Drop original lat/lon to avoid multicollinearity

# Separates non numerical columns from numerical ones
non_numerical_columns = ['saleEstimate_currentPrice', 'tenure', 'propertyType', 'currentEnergyRating']

# Process df1lr (for Linear Regression): scale and encode
numerical_features_lr = [col for col in df1lr.columns if col not in non_numerical_columns]
scaler_lr = StandardScaler()
df1lr[numerical_features_lr] = scaler_lr.fit_transform(df1lr[numerical_features_lr])
categorical_cols_lr = df1lr.select_dtypes(include='object').columns
df1lr = pd.get_dummies(df1lr, columns=categorical_cols_lr, drop_first=True)

target_column = 'saleEstimate_currentPrice'
x_lr = df1lr[[col for col in df1lr.columns if col != target_column]].copy()
y_lr = df1lr[target_column].copy()
x_train_lr, x_test_lr, y_train_lr, y_test_lr = train_test_split(x_lr, y_lr, test_size=0.2, random_state=42)

# Process df1 (for tree models: Random Forest, XGBoost): keep raw lat/lon, scale and encode
numerical_features = [col for col in df1.columns if col not in non_numerical_columns]
scaler = StandardScaler()
df1[numerical_features] = scaler.fit_transform(df1[numerical_features])
categorical_cols_to_encode = df1.select_dtypes(include='object').columns
df1 = pd.get_dummies(df1, columns=categorical_cols_to_encode, drop_first=True)

features = [col for col in df1.columns if col != target_column]
x = df1[features].copy()
y = df1[target_column].copy()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 1: Dataset 1.csv")
print("=" * 60)
print("\nFirst 10 rows:")
display(df1.head(10))
print("\nData types:")
print(df1.dtypes)
print("\nMissing values:")
print(df1.isnull().sum())
print("\nTarget variable (saleEstimate_currentPrice) statistics:")
print(df1['saleEstimate_currentPrice'].describe())

In [ ]:
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np
# Inverse the log transformation to its original scale

def rmse_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate RMSE on the original scale
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

def mae_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate MAE on the original scale
    return mean_absolute_error(y_true_original, y_pred_original)

def r2_original_scale(y_true, y_pred):
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate R2 on the original scale
    return r2_score(y_true_original, y_pred_original)

# Create scorers that can be used with cross_val_score
neg_rmse_original_scorer = make_scorer(rmse_original_scale, greater_is_better=False)
neg_mae_original_scorer = make_scorer(mae_original_scale, greater_is_better=False)
r2_original_scorer = make_scorer(r2_original_scale, greater_is_better=True)

### Linear Regression for Dataset 1

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# Train baseline Linear Regression
lr = LinearRegression()
lr.fit(x_train_lr, y_train_lr)
y_pred_lr = lr.predict(x_test_lr)

# Inverse log transform to get actual prices
y_test_lr_actual = np.exp(y_test_lr)
y_pred_lr_actual = np.exp(y_pred_lr)

# Evaluate on original scale
rmse_lr = np.sqrt(mean_squared_error(y_test_lr_actual, y_pred_lr_actual))
mae_lr = mean_absolute_error(y_test_lr_actual, y_pred_lr_actual)
r2_lr = r2_score(y_test_lr_actual, y_pred_lr_actual)

print("=" * 50)
print("BASELINE: Linear Regression on Dataset 1.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_lr:,.2f}")
print(f"MAE:\t${mae_lr:,.2f}")
print(f"R²:\t{r2_lr:.4f}")
print(f"\nCross-validation RMSE:\t${-cross_val_score(lr, x_lr, y_lr, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(lr, x_lr, y_lr, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(lr, x_lr, y_lr, cv=5, scoring=r2_original_scorer).mean():.4f}")

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test_lr_actual, y_pred_lr_actual, alpha=0.4, s=15)
plt.plot([0, y_test_lr_actual.max()], [0, y_test_lr_actual.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Baseline Linear Regression: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Fetching Feature Names and Coefficients
feature_names = x_train_lr.columns
coefficients = lr.coef_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_lr = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# Using the Absolute Function before Sorting the Coefficients in Descending Order
feature_importance_lr['Abs_Coefficient'] = np.abs(feature_importance_lr['Coefficient'])
top_10_features_lr = feature_importance_lr.sort_values(by='Abs_Coefficient', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_lr,
    x='Coefficient',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (Linear Regression)', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_lr[['Feature', 'Coefficient']])

# Analysis

The R² of the Linear Regression model is 0.6694, which implies that the model explains about 67% of the price movement in the property prices. An average error of £163,665 provides a decent description of the average error of prediction and the root mean squared error is much larger at £295,221. The difference between MAE and RMSE indicates that there is a set of large errors, further confirmed by the scatter plot, which demonstrates that most of the lower-priced properties (less than about £1.5M) are following fairly well along the diagonal, but there exists a cluster of extreme overestimates at the higher end, with predictions of properties exceeding £10M occurring for property values actually worth £2.5-3.5M. This extreme volatility is a structural weakness of linear regression in the case of a skewed price distribution towards the right. The results of cross-validation are in line with the results: CV RMSE of £301,510, CV MAE of £164,073, and CV R-Squared of 0.6586 indicating that the model is not over-fitting.

The most influential feature is floorAreaSqM with the largest positive coefficient of 0.289 in the log-transformed space. A one standard deviation increase in floor area is linked to a more or less 29% increase in the predicted price, which is logical, since larger properties are associated with having significantly higher sale prices.

Property type dummies dominate the negative side of the chart. The propertyTypePurpose Built Flat has the lowest coefficient (−0.304), then the propertytype Converted Flat ( -0.199), propertytype Flat/Maisonette ( -0.192), propertytype Semi-Detached Property ( -0.167), and propertytype End Terrace Property (-0.158). All these are sold at a huge discount as compared to the excluded baseline category (detached houses), which are in line with known UK market trends.

Two engineered geographic features appear in the top 10. lat_lon_interaction (latitude × longitude) was created so that the model would have only one term in which it could compute the combined geographic location of a property, as opposed to considering latitude and longitude as merely additive features. Its negative coefficient (-0.152) is a general measure of the pricing gradient across the spatial range of the dataset. dist_to_center (Euclidean distance to the geographic center of the dataset) was also designed to allow let the model interact with distance from the densest, usually highest-priced portion of the sample; the negative coefficient (-0.124) of dist_to_center confirms that properties further from the center are associated with lower predicted prices.

tenureShared has a significant positive coefficient (+0.137), probably due to the composition bias in the data, being that shared-ownership properties in this dataset may skew towards more expensive urban areas, rather than indicating a genuine tenure premium (i.e. how much more or less buys will pay for one tenure type). The other type of property with a small positive coefficient is propertytypeEnd Terrace Bungalow which has a slight premium over some of the flat categories which make up the negative tail.

The main weakness of this model lies in its weak calibration at the upper end of the price distribution: the RMSE is almost twice the MAE, which is a clear indication of large outlier errors. Its R², of 0.6694 and cross-validated R² of 0.6586 represents a significant proportion of variance, but the remaining 33% is due to non-linear interactions that the model cannot capture. These interactions and the high-end prediction error should be better handled by non-linear models like the Random Forest.

# Random Forest Regression for Dataset 1

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

rf = RandomForestRegressor(n_estimators=203, n_jobs=5, max_features=0.91, max_depth=50, random_state=42)
rf.fit(x_train, y_train)
y_pred_rf = rf.predict(x_test)

# Inverse log transform to get actual prices
y_test_actual = np.exp(y_test)
y_pred_rf_actual = np.exp(y_pred_rf)

# Evaluate on original scale
rmse_rf = np.sqrt(mean_squared_error(y_test_actual, y_pred_rf_actual))
mae_rf = mean_absolute_error(y_test_actual, y_pred_rf_actual)
r2_rf = r2_score(y_test_actual, y_pred_rf_actual)

print("=" * 50)
print("Random Forest Regressor on Dataset 1.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_rf:,.2f}")
print(f"MAE:\t${mae_rf:,.2f}")
print(f"R²:\t{r2_rf:.4f}")

print(f"\nCross-validation RMSE:\t${-cross_val_score(rf, x, y, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(rf, x, y, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(rf, x, y, cv=5, scoring=r2_original_scorer).mean():.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test_actual, y_pred_rf_actual, alpha=0.4, s=15)
plt.plot([0, y_test_actual.max()], [0, y_test_actual.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Random Forest Regressor: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Fetching Feature Names and Importances from Random Forest
feature_names = x_train.columns
importances = rf.feature_importances_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_rf = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sorting Feature Importances in Descending Order
top_10_features_rf = feature_importance_rf.sort_values(by='Importance', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_rf,
    x='Importance',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (Random Forest)', fontsize=15)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_rf[['Feature', 'Importance']])

# Analysis

The story of what drives property prices as explained by the Random Forest model is quite different than the Linear Regression. The continuous variables that dominate the majority of this feature importance chart are: floorAreaSqM (~0.62), longitude (~0.16) and latitude (~0.12). Combining the three features, the overall predictive power of the model is computed to be around 90 percent, whereas all other features have only small significant value.

The most significant feature, which is floorareaSqM, is not only consistent with the results of the Linear Regression but also makes sense in the real world, being that property size is the one of, if not the most important characteristic to predict price. But the most interesting thing about the Random Forest model is the prominence of longitude and latitude. These geographic coordinates were hardly visible in the most popular features of the Linear Regression as the impact of location on price is very non-linear. For instance, a property in central London (specific longitude/latitude ranges) can be worth 1.11x the price of a similar property elsewhere, as indicated in the code block below. The Random Forest can capture these non-linear spatial patterns through its tree-based splits, effectively learning "if longitude is between X and Y and latitude is between A and B, then the property is in a premium area."

The other attributes such as "bedrooms, bathrooms, tenure Freehold, living Rooms and other types of property have lower contribution of less than 1-2 percent each. This does not imply they are irrelevant, but that once the floor area and location is known, they contribute a marginal amount to the predictive signal. This is in part due to the fact that much of the information that will be given by bedrooms and bathrooms is already encoded in the floor area (larger homes naturally have more rooms).

The scatter plot proves the superiority of the Random Forest: the predictions follow the diagonal line far better through the whole price range, even the higher bracket where Linear Regression failed. The RMSE is much lower than the Linear Regression and the difference between the RMSE and MAE is much less, meaning that the model no longer suffers the extreme outlier errors that affected the linear model on the high end.

The general lesson learned is that in this case, is that property prices depend mostly on size and location, and the non-linear geographic pricing patterns that Linear Regression can not identify can be captured by the Random Forest. The fact that the model only uses three features to make 94% of its predictions also indicates that a finer location-based features (e.g. proximity to transport, school ratings, neighbourhood crime rates) may be useful in this dataset to give it additional predictive power over raw coordinates.

# Supporting Evidence: Central London vs. Non-Central London Price Disparity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle

# Reload raw data to work with actual (untransformed) prices
df_raw = df1.copy()
df_raw.dropna(subset=['saleEstimate_currentPrice', 'latitude', 'longitude'], inplace=True)

# Covers: Westminster, City of London, Kensington & Chelsea, Camden core
CENTRAL_LON_MIN, CENTRAL_LON_MAX = -0.18, 0.05
CENTRAL_LAT_MIN, CENTRAL_LAT_MAX =  51.48, 51.54

df_raw['is_central'] = (
    (df_raw['longitude'] >= CENTRAL_LON_MIN) & (df_raw['longitude'] <= CENTRAL_LON_MAX) &
    (df_raw['latitude']  >= CENTRAL_LAT_MIN) & (df_raw['latitude']  <= CENTRAL_LAT_MAX)
)

central = df_raw[df_raw['is_central']]['saleEstimate_currentPrice']
non_central = df_raw[~df_raw['is_central']]['saleEstimate_currentPrice']

mean_central = central.mean()
mean_non_central = non_central.mean()
ratio = mean_central / mean_non_central

print("=" * 65)
print("Central London vs. Non-Central London: Price Comparison")
print("=" * 65)
print(f"Central London properties:      {len(central):,}")
print(f"Non-Central London properties:  {len(non_central):,}")
print(f"\nMean price - Central London:      £{mean_central:,.0f}")
print(f"Mean price - Non-Central London:  £{mean_non_central:,.0f}")
print(f"\nPrice ratio (Central / Non-Central): {ratio:.2f}x")

print("\nPrice percentile breakdown:")
for p in [25, 50, 75, 90]:
    c_val  = np.percentile(central, p)
    nc_val = np.percentile(non_central, p)
    print(f"{p}th percentile - Central: £{c_val:.0f} | "
          f"Non-Central: £{nc_val:.0f} | Ratio: {c_val/nc_val:.2f}x")

# Hyperparameter Tuning for XGBoost

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [6, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.5]
}

xgb_model = XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=5)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=10, 
    scoring=neg_mae_original_scorer, 
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train, y_train)

print(f"Best Parameters: {random_search.best_params_}")
print(f"Best CV MAE (Original Scale): ${-random_search.best_score_:,.2f}")

best_xgb = random_search.best_estimator_
y_pred_best = best_xgb.predict(x_test)

y_test_actual = np.exp(y_test)
y_pred_best_actual = np.exp(y_pred_best)

mae_best = mean_absolute_error(y_test_actual, y_pred_best_actual)
print(f"Final Test MAE with Optimized Model: ${mae_best:,.2f}")

# XGBoost Regressor for Dataset 1

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import matplotlib.pyplot as plt

# Train XGBoost Regressor
xgb = XGBRegressor(n_estimators=200, subsample=0.7, max_depth=15, learning_rate=0.1, gamma=0, colsample_bytree=0.7, objective="reg:squarederror", random_state=42, n_jobs=5)
xgb.fit(x_train, y_train)
y_pred_xgb = xgb.predict(x_test)

# Inverse log transform to get actual prices
y_pred_xgb_actual = np.exp(y_pred_xgb)
y_test_actual = np.exp(y_test)

# Evaluate on original scale
rmse_xgb = np.sqrt(mean_squared_error(y_test_actual, y_pred_xgb_actual))
mae_xgb = mean_absolute_error(y_test_actual, y_pred_xgb_actual)
r2_xgb = r2_score(y_test_actual, y_pred_xgb_actual)

print("=" * 50)
print("XGBOOST: XGBoost Regressor on Dataset 1.csv")
print("=" * 50)
print(f"RMSE:\t${rmse_xgb:,.2f}")
print(f"MAE:\t${mae_xgb:,.2f}")
print(f"R²:\t{r2_xgb:.4f}")

print(f"\nCross-validation RMSE:\t${-cross_val_score(xgb, x, y, cv=5, scoring=neg_rmse_original_scorer).mean():,.2f}")
print(f"Cross-validation MAE:\t${-cross_val_score(xgb, x, y, cv=5, scoring=neg_mae_original_scorer).mean():,.2f}")
print(f"Cross-validation R²:\t{cross_val_score(xgb, x, y, cv=5, scoring=r2_original_scorer).mean():.4f}")

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test_actual, y_pred_xgb_actual, alpha=0.4, s=15)
plt.plot([0, y_test_actual.max()], [0, y_test_actual.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('XGBoost Regressor: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

# Feature Importance Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Fetching Feature Names and Importances from XGBoost
feature_names = x_train.columns
importances = xgb.feature_importances_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_xgb = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sorting Feature Importances in Descending Order
top_10_features_xgb = feature_importance_xgb.sort_values(by='Importance', ascending=False).head(10)

# Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_10_features_xgb,
    x='Importance',
    y='Feature',
)
plt.title('Top 10 Features Driving Property Prices (XGBoost)', fontsize=15)
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(top_10_features_xgb[['Feature', 'Importance']])

### Supporting Evidence: Feature Importance Distribution - XGBoost vs Random Forest:

In [ ]:
rf_top3 = top_10_features_rf.head(3)['Importance'].sum()
xgb_top3 = top_10_features_xgb.head(3)['Importance'].sum()

print(f"RF top-3 cumulative importance:      {rf_top3:.1%}")
print(f"XGBoost top-3 cumulative importance: {xgb_top3:.1%}")

### Supporting Evidence: Sale Price by Tenure Type

In [ ]:
# Create a copy of the data
df = df1.copy()

def to_currency(value): # Function for formatting numbers as currency
    return f"£{value:,.0f}"

# Group by tenure and get the stats
stats = df.groupby('tenure')['saleEstimate_currentPrice'].describe()[['count', 'mean', '50%']]
stats.columns = ['Count', 'Mean', 'Median']

# Sort by mean
stats = stats.sort_values(by='Mean', ascending=False)

stats['Mean'] = stats['Mean'].map(to_currency)
stats['Median'] = stats['Median'].map(to_currency)

print("Mean and Median Sale Price by Tenure Type:")
print(stats)

### Supporting Evidence: Mean Sale Price by Bedroom Count

In [ ]:
df = df1.copy()
df_filtered = df[(df['bedrooms'] >= 1) & (df['bedrooms'] <= 8)]

# Group and get the count and mean
bed_stats = df_filtered.groupby('bedrooms')['saleEstimate_currentPrice'].describe()[['count', 'mean']]

bed_stats.columns = ['Count', 'Mean']

bed_stats['Mean'] = [f"£{x:,.0f}" for x in bed_stats['Mean']]

print("Mean Sale Price by Bedroom Count (1-8 bedrooms):")
print(bed_stats)

### Supporting Evidence: Correlation Between floorAreaSqM, bedrooms, and bathrooms

In [ ]:
corr = df_raw[['floorAreaSqM', 'bedrooms', 'bathrooms']].dropna().corr().round(3)
print(corr.to_string())

# Analysis

XGBoost improves on the Random Forest across every metric: RMSE drops from $58,467.10 to $50,734.44, MAE drops from $21,370.12 to $18,792.24, and R² improves from 0.9870 to 0.9902. Cross-validation results are close to the test results in both models, so neither model is overfitting.

While the key difference between models is the relative importance of features. In the Random Forest, floorAreaSqM was the most important feature with importance at ~0.62, followed by longitude (~0.16) and latitude (~0.12). In XGBoost, tenure_Leasehold takes the top spot at ~0.278, with floorAreaSqM dropping to third at ~0.113, and the geographic features slipping to seventh and ninth. bedrooms and bathrooms, which each had less than 3% importance in the Random Forest, now stand at ~0.12 and ~0.10 respectively.

A direct consequence of this is that importance is more evenly spread across features in XGBoost than in the Random Forest. The top three features in the Random Forest explain almost 90% of the overall importance, whereas the top three in XGBoost explain almost 51% (see code cell below).

The scatter plot for XGBoost follows the diagonal tightly across the full price range, consistent with the lower RMSE. The gap between RMSE and MAE, which reflects sensitivity to high-end outliers, narrows slightly from ~$37,097 in the Random Forest to ~$31,942 in XGBoost, meaning XGBoost handles those extreme prices marginally better.

The XGBoost scatter plot is very close to the diagonal line across the board, in line with the lowered RMSE. The difference between RMSE and MAE, which reflects sensitivity to high-end outliers, shrinks from ~$37,097 to ~$31,942 from the Random Forest to XGBoost, meaning XGBoost deals with those high prices slightly better.

tenure_Leasehold being at the top is consistent with the UK housing market. In England and Wales, leasehold property and freehold property are essentially in different price brackets. Leasehold is mainly for flat and apartments, while freehold is mainly for houses. As we see in the evidence code block below, the average price of a Freehold property is £1,207,080 whereas for a Leasehold it's £720,241, a difference of roughly £487,000. This tells us there's a large, high-value split in the data just from tenure type alone.

This is supported by the fact that tenureFreehold comes in at fifth place with a value of ~0.073. tenureLeasehold and tenure_Freehold account for about 35% of the total feature importance of the model, so just the type of tenure accounts for over a third of the XGBoost model's predictive power.

The rise of bedrooms to the second position and bathrooms to the fourth follows once tenure has separated property types. For any given tenure, the number of bedrooms is the most direct indicator of how much a property offers, and translates directly to price. The next code block below confirms a steady stepped function relationship: the mean sale price increases from £486,842 for 1-bedroom properties to £4,861,663 for 8-bedroom properties.

The transition from Random Forest to XGBoost represents a shift from a model that is strongly influenced by continuous physical size (floorAreaSqM) to one that favours legal and spatial classes (tenure, bedrooms). This would imply that while size is a key determinant of value, the utility and legality of the square footage offers a more "pure" signal to be used for gradient descent to minimize loss, particularly when penalising for outlier behaviour.

floorAreaSqM is still number three but has fallen from its position of prominence in the Random Forest. One explanation is that floor area and the number of bedrooms share information, bigger properties typically have more bedrooms and bathrooms. The last evidence code block reports a 0.795 Pearson correlation between floorAreaSqM and bedrooms, and a 0.676 Pearson correlation between floorAreaSqM and bathrooms, suggesting that once bedrooms and bathrooms have captured this signal, floorAreaSqM contributes less additional gain in the splits.